# Analysis of Bank Marketing Data

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("../data/bank_marketing_data.csv", sep=";")
# Drop the unnamed index column
df = df.drop(columns=[df.columns[0]])
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,target,post_campaign_action
0,30,unemployed,married,primary,no,1787.0,no,no,cellular,19,oct,79,1,-1,0,unknown,no,0
1,33,services,married,secondary,no,4789.0,yes,yes,cellular,11,may,220,1,339,4,failure,no,0
2,35,management,single,tertiary,no,1350.0,yes,no,cellular,16,apr,185,1,330,1,failure,no,0
3,30,management,married,tertiary,no,1476.0,yes,yes,unknown,3,jun,199,4,-1,0,unknown,no,0
4,59,blue-collar,married,secondary,no,0.0,yes,no,unknown,5,may,226,1,-1,0,unknown,no,0


In [3]:
print(f"Dataset shape: {df.shape}")
print(f"\nColumn dtypes:\n{df.dtypes}")

Dataset shape: (4566, 18)

Column dtypes:
age                       int64
job                      object
marital                  object
education                object
default                  object
balance                 float64
housing                  object
loan                     object
contact                  object
day                       int64
month                    object
duration                  int64
campaign                  int64
pdays                     int64
previous                  int64
poutcome                 object
target                   object
post_campaign_action      int64
dtype: object


## Duplicates

In [4]:
n_dupes = df.duplicated().sum()
if n_dupes > 0:
    df = df.drop_duplicates()
    print(f"Removed {n_dupes} duplicates. New shape: {df.shape}")

Removed 23 duplicates. New shape: (4543, 18)


## Missing Values

In [5]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"Count": missing, "Percent": missing_pct})
missing_df = missing_df[missing_df["Count"] > 0]
missing_df

,Count,Percent
job,88,1.94
education,137,3.02
balance,135,2.97
contact,133,2.93


### Impute Missing Values
Missing numeric values are imputed using the median value of the column and missing categorical values are imputed using the mode value of the column.

In [6]:
num_data = df.select_dtypes(include=np.number)
cat_data = df.select_dtypes(exclude=np.number)

num_cols_with_na = num_data.columns[num_data.isnull().any()]
cat_cols_with_na = cat_data.columns[cat_data.isnull().any()]

for col in num_cols_with_na:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f"{col}: filled NaNs with median value ({median_val})")

for col in cat_cols_with_na:
    mode_val = df[col].mode()[0]
    df[col] = df[col].fillna(mode_val)
    print(f"{col}: filled NaNs with mode value ('{mode_val}')")

balance: filled NaNs with median value (447.0)
job: filled NaNs with mode value ('management')
education: filled NaNs with mode value ('secondary')
contact: filled NaNs with mode value ('cellular')


## Descriptive Statistics

### Numeric Data Summary

In [7]:
df.describe().round(2)

,age,balance,day,duration,campaign,pdays,previous,post_campaign_action
count,4543.00,4543.00,4543.00,4543.00,4543.00,4543.00,4543.00,4543.00
mean,41.22,1406.20,15.90,263.96,2.79,39.75,0.55,0.38
std,10.81,3006.74,8.25,259.60,3.11,100.10,1.71,0.49
min,19.00,-3313.00,1.00,4.00,1.00,-1.00,0.00,0.00
25%,33.00,76.00,9.00,104.00,1.00,-1.00,0.00,0.00
50%,39.00,447.00,16.00,185.00,2.00,-1.00,0.00,0.00
75%,49.00,1427.00,21.00,330.00,3.00,-1.00,0.00,1.00
max,131.00,71188.00,31.00,3025.00,50.00,871.00,25.00,1.00


Max age 131?

Max duration is 3025 (I assume in seconds) which is a 50 min call. Not unreasonable...?

### Categorical Data Summary

In [8]:
cat_cols = df.select_dtypes(exclude=np.number).columns
for col in cat_cols:
    vc = df[col].value_counts().to_dict()
    print(f"{col}: {len(vc)} Unique values")
    print(f"{vc}\n")

job: 12 Unique values
{'management': 1026, 'blue-collar': 910, 'technician': 743, 'admin': 462, 'services': 402, 'retired': 225, 'self-employed': 175, 'entrepreneur': 155, 'unknown': 127, 'unemployed': 125, 'housemaid': 112, 'student': 81}

marital: 3 Unique values
{'married': 2810, 'single': 1203, 'divorced': 530}

education: 4 Unique values
{'secondary': 2397, 'tertiary': 1303, 'primary': 665, 'unknown': 178}

default: 2 Unique values
{'no': 4467, 'yes': 76}

housing: 2 Unique values
{'yes': 2572, 'no': 1971}

loan: 2 Unique values
{'no': 3850, 'yes': 693}

contact: 3 Unique values
{'cellular': 2877, 'unknown': 1385, 'telephone': 281}

month: 12 Unique values
{'may': 1406, 'jul': 706, 'aug': 638, 'jun': 534, 'nov': 389, 'apr': 294, 'feb': 225, 'jan': 149, 'oct': 80, 'sep': 53, 'mar': 49, 'dec': 20}

poutcome: 4 Unique values
{'unknown': 3772, 'failure': 464, 'other': 187, 'success': 120}

target: 2 Unique values
{'no': 4019, 'yes': 524}



## Target Distribution

In [9]:
target_vc = df["target"].value_counts()
target_vc

target
no     4019
yes     524
Name: count, dtype: int64

### Imbalance Ratio

In [10]:
print(round((target_vc.get("no") / target_vc.get("yes")), 2))

7.67


# Feature Engineering

In [11]:
df["target"] = df["target"].map({"yes": 1, "no": 0})

# pdays should be binned to something more meaningful
bins = [-2, 0, 7, 30, np.inf]
labels = ["never", "within_a_week", "within_a_month", "over_a_month"]

df["pcontacted"] = pd.cut(df["pdays"], bins=bins, labels=labels)
df.drop(columns=["pdays"], inplace=True)

# Drop "day" and "month"? Perhaps there is a relationship between
# those and the target variable, e.g. no one will sign up if phoned
# on the weekend. But we won't now the exact date when someone will
# be contacted. So we can't use this information for training.
df.drop(columns=["day", "month"], inplace=True)


df.drop(columns=["contact"], inplace=True)

# Data leakage
df.drop(columns=["post_campaign_action", "duration"], inplace=True)
df.head()

,age,job,marital,education,default,balance,housing,loan,campaign,previous,poutcome,target,pcontacted
0,30,unemployed,married,primary,no,1787.0,no,no,1,0,unknown,0,never
1,33,services,married,secondary,no,4789.0,yes,yes,1,4,failure,0,over_a_month
2,35,management,single,tertiary,no,1350.0,yes,no,1,1,failure,0,over_a_month
3,30,management,married,tertiary,no,1476.0,yes,yes,4,0,unknown,0,never
4,59,blue-collar,married,secondary,no,0.0,yes,no,1,0,unknown,0,never


### Encode Features

In [12]:
cat_features = df.select_dtypes(exclude=np.number).columns.tolist()
df_encoded = pd.get_dummies(df, columns=cat_features, drop_first=False, dtype=int)
df_encoded.shape

(4543, 38)

In [13]:
feature_cols = [c for c in df_encoded.columns if c != "target"]
X = df_encoded[feature_cols]
y = df_encoded["target"]

## Prepare Train/Test Data

In [14]:
import random

from sklearn.model_selection import train_test_split


SEED = 24

random.seed(SEED)
np.random.seed(SEED)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

### Handle class imbalance with SMOTE 

In [15]:
from imblearn.over_sampling import SMOTE

# Logistic Regression will beneft from SMOTE.
# Random Forest and XGBoost are more robust to imbalanced data,
# but may also benefit. Tuning experiments are needed to confirm.

smote = SMOTE(random_state=SEED)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"Orginal training set: {X_train.shape[0]} samples")
print(f"Orginal target distribution:\n{y_train.value_counts()}")

print(f"Resampled training set: {X_train_res.shape[0]} samples")
print(f"Resampled target distribution:\n{y_train_res.value_counts()}")

Orginal training set: 3634 samples
Orginal target distribution:
target
0    3215
1     419
Name: count, dtype: int64
Resampled training set: 6430 samples
Resampled target distribution:
target
0    3215
1    3215
Name: count, dtype: int64


### Feature Scaling

In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test)

# Keep unscaled versions for tree-based models
X_train_tree = X_train_res.values
X_test_tree = X_test.values

### RF

Use a RF to get an idea of performace

In [17]:
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.ensemble import RandomForestClassifier


rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=7,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=SEED,
    n_jobs=-1,
)

rf_model.fit(X_train_tree, y_train_res)
rf_pred = rf_model.predict(X_test_tree)
rf_prob = rf_model.predict_proba(X_test_tree)[:, 1]

print(f"Training accuracy: {rf_model.score(X_train_tree, y_train_res):.4f}")
print(f"Test accuracy:     {accuracy_score(y_test, rf_pred):.4f}")
print(f"Test ROC-AUC:      {roc_auc_score(y_test, rf_prob):.4f}")

Training accuracy: 0.9123
Test accuracy:     0.8922
Test ROC-AUC:      0.6965


In [18]:
from sklearn.model_selection import cross_val_score, StratifiedKFold


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=13)
rf_cv = cross_val_score(rf_model, X_train_tree, y_train_res, cv=cv, scoring="roc_auc")

print(f"Random Forest:  {rf_cv.mean():.4f} ± {rf_cv.std():.4f}")

Random Forest:  0.9511 ± 0.0065


## Plots

In [19]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [20]:
# ── Plot: Target distributions ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
target_vc.plot(kind="bar", color=["#3498db", "#e74c3c"], edgecolor="black", ax=ax)
ax.set_title("Target Distribution", fontweight="bold")
ax.set_ylabel("Count")
ax.set_xlabel("Target")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for i, v in enumerate(target_vc.values):
    ax.text(i, v + 30, str(v), ha="center", fontweight="bold")
plt.tight_layout()
plt.savefig("./plots/01_target_distribution.png", dpi=300)
plt.close()

# ── Plot: Numeric distributions ───────────────────────────────────────────
num_cols = df.select_dtypes(include=np.number).columns.tolist()
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.ravel()
for i, col in enumerate(num_cols[:9]):
    sns.histplot(df[col], kde=True, ax=axes[i], color="#3498db", edgecolor="black")
    axes[i].set_title(col, fontweight="bold")
for j in range(i + 1, 9):
    axes[j].set_visible(False)
plt.suptitle("Numeric Feature Distributions", fontweight="bold", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("./plots/02_numeric_distributions.png", dpi=300)
plt.close()

# ── Plot: Correlation heatmap ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Correlation Heatmap (Numeric Features)", fontweight="bold")
plt.tight_layout()
plt.savefig("./plots/03_correlation_heatmap.png", dpi=300)
plt.close()

# ── Plot: Categorical features vs target ──────────────────────────────────
cat_cols_plot = [c for c in cat_cols if c != "target" and c != "post_campaign_action"]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()
for i, col in enumerate(cat_cols_plot[:6]):
    ct = pd.crosstab(df[col], df["target"], normalize="index") * 100
    ct.plot(
        kind="bar",
        stacked=True,
        ax=axes[i],
        color=["#3498db", "#e74c3c"],
        edgecolor="black",
    )
    axes[i].set_title(f"{col} vs target", fontweight="bold")
    axes[i].set_ylabel("Percentage")
    axes[i].legend(title="target", loc="upper right")
    axes[i].tick_params(axis="x", rotation=45)
for j in range(i + 1, 6):
    axes[j].set_visible(False)
plt.suptitle(
    "Categorical Features vs Target (% Stacked)", fontweight="bold", fontsize=14, y=1.01
)
plt.tight_layout()
plt.savefig("./plots/04_categorical_vs_target.png", dpi=300)
plt.close()

# ── Plot: Numeric features box-plots by target ───────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.ravel()
for i, col in enumerate(num_cols[:9]):
    sns.boxplot(data=df, x="target", y=col, ax=axes[i], palette=["#3498db", "#e74c3c"])
    axes[i].set_title(col, fontweight="bold")
for j in range(i + 1, 9):
    axes[j].set_visible(False)
plt.suptitle("Numeric Features by Target", fontweight="bold", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("./plots/05_numeric_boxplots_by_target.png", dpi=300)
plt.close()

model_name = "Random Forest"
importances = rf_model.feature_importances_
idx = np.argsort(importances)[-15:]
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(range(len(idx)), importances[idx], color="#3498db", edgecolor="black")
ax.set_yticks(range(len(idx)))
ax.set_yticklabels([feature_cols[i] for i in idx])
ax.set_title(f"Top 15 Feature Importances — {model_name}", fontweight="bold")
ax.set_xlabel("Importance")
plt.tight_layout()
safe_name = model_name.lower().replace(" ", "_")
plt.savefig(f"./plots/06_feature_importance_{safe_name}.png", dpi=300)
plt.close()

/tmp/ipykernel_171524/323982377.py:62: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="target", y=col, ax=axes[i],
/tmp/ipykernel_171524/323982377.py:62: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="target", y=col, ax=axes[i],
/tmp/ipykernel_171524/323982377.py:62: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="target", y=col, ax=axes[i],
/tmp/ipykernel_171524/323982377.py:62: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `